# Synthesize Simple Summary from Relevant Paper

## Step 0: Environment Setup

In [1]:
from dotenv import load_dotenv
import os
from langchain_neo4j import Neo4jGraph
from libs import create_vector_index, get_entities,embed_entity, vector_search,enhanced_chunk_finder
import pandas as pd
from conn import connect2Googlesheet
from openai import OpenAI
from synthlibs import graph_retriever, parse_relationships, final_context_builder,synthesize_response,load_prompt_template,get_firstk_chunks

# plot figures in high resolution
%config InlineBackend.figure_format = 'retina'
# Force reload of the .env file
try:
    load_dotenv()
    print("The .env file has been loaded")
except Exception as e:
    print("The .env file has not been loaded. Reported error: ", e)

The .env file has been loaded


In [2]:
# Connect to Neo4j database
try:
    graph = Neo4jGraph(
        url=os.getenv("NEO4J_URL"),
        username=os.getenv("NEO4J_USERNAME"),
        password=os.getenv("NEO4J_PASSWORD")
    )
    print("Connected to Neo4j database successfully.")
except ValueError as e:
    print(f"Could not connect to Neo4j database: {e}")
    
# Check if the entities index exists
index_name = "entities"
query = "SHOW INDEXES YIELD name, type WHERE type = 'VECTOR' AND name = $index_name"

result = graph.query(query, params={"index_name": index_name})
if result:
    print("The 'entities' index already exists.")
else:
    create_vector_index(graph, "entities")

Connected to Neo4j database successfully.
The 'entities' index already exists.


## Step 1: Connect to Google Sheet

In [7]:
spreadsheet = connect2Googlesheet()

# Select the worksheet: relevance
Q_worksheet = spreadsheet.get_worksheet(2)  
# Select the worksheet: revised summary
Sum_worksheet = spreadsheet.get_worksheet(6) 

# Get all records as a list of dictionaries
ques_data = Q_worksheet.get_all_records()
sum_data = Sum_worksheet.get_all_records()

# Convert to Pandas DataFrame
df_MedQ = pd.DataFrame(ques_data)
df_MedSum = pd.DataFrame(sum_data)

# Display the first few rows of both DataFrames
display(df_MedQ.head())
display(df_MedSum.head())

,condition,number,docs,Question,Mahmud's Note,status,comments,
0,ARDS,1,ACURASYS,Does early administration of neuromuscular blo...,Like,,,
1,ARDS,2,ACURASYS,Do patients with severe ARDS being treated wit...,Replace,fixed,,
2,ARDS,3,ROSE,"In patients with moderate to severe ARDS, does...",Maybe this question: In patients with moderate...,fixed,,
3,ARDS,4,ROSE,Do patients with moderate-to-severe ARDS have ...,Local question (not sure if this is the aim of...,fixed,Wrong concept since PEEP by itself is mandator...,Does the use of neuromuscular blockers in pati...
4,ARDS,5,FACTT,"Among patients with ALI/ARDS, does a conservat...",Local question (not sure if this is the aim of...,fixed,Check if studies defined conservative by CVP <...,


,Doc No,Confirmed by,Topic,Name,Study Design and Methodology,Study Population,Interventions,Comparator,Outcomes,Strengths and Weaknesses,Key Findings and Conclusion,JSON Mahmoud,JSON Bob's Model
0,1,Keon,ARDS,ACURASYS,"- Multicenter, randomized, double-blind, place...",The study enrolled 340 patients from 20 ICUs i...,- Cisatracurium Group:\n - 3-ml rapid IV infu...,- Placebo group receiving identical IV infusio...,Primary Outcome:\n- 90-day in-hospital mortali...,Strengths:\n- Multicenter design\n- Blinded ra...,Key Findings:\n- Adjusted 90-day survival impr...,"{\n ""design"": ""Multicenter, randomized, dou...","{\n ""design"": ""Multicenter, double-blind tria..."
1,2,Keon,ARDS,ROSE,"- Multicenter, unblinded, randomized clinical ...",- Patients with moderate-to-severe ARDS\n- Cri...,- 48-hour continuous infusion of cisatracurium...,- Usual care approach\n- No routine neuromuscu...,Primary Outcome:\n- In-hospital death from any...,Strengths:\n- Multicenter design\n- High proto...,Key Findings:\n No significant difference in 9...,"{\n ""design"": ""Multicenter, unblinded, rand...","{\n ""design"": ""Multicenter, unblinded, random..."
2,3,Keon,ARDS,FACTT,"- Prospective, randomized clinical trial\n- Co...","- 1,001 patients with acute lung injury (503 i...",Conservative Strategy:\n- Lower fluid intake\n...,The two fluid management strategies were compa...,Primary Outcome:\n- Death at 60 days\n\nSecond...,Strengths:\n- Randomized design\n- Multi-cente...,Key Findings:\n- No significant difference in ...,"{\n ""design"": ""Prospective, randomized clin...","{\n ""design"": ""Prospective, randomized clin..."
3,4,Keon,ARDS,ARDSNet,"- Multicenter, randomized controlled trial\n- ...",- Total Participants: 861 patients\n- Inclusio...,- Lower Tidal Volume Group:\n - Initial tidal...,- Traditional Tidal Volume Group:\n - Initial...,Primary Outcomes:\n- Mortality before hospital...,Strengths:\n- Multicenter design\n- Robust ran...,Key Findings:\n- Mortality reduced by 22% in l...,"{\n ""design"": ""Multicenter, randomized cont...","{\n ""design"": ""Randomized controlled trial"",\..."
4,5,Keon,ARDS,PROSEVA,"- Multicenter, prospective, randomized, contro...",- 466 patients with severe ARDS\n- Inclusion c...,- Prone group (237 patients):\n - Turned to p...,- Supine group (229 patients):\n - Remained i...,Primary Outcome:\n- 28-day mortality\n\nSecond...,Strengths:\n- Multicenter design\n- Standardiz...,Key Findings:\n- 28-day mortality: 16.0% (pron...,"{\n ""design"": ""Multicenter, prospective, ra...","{\n ""design"": ""Prospective, multicenter, rand..."


## Step 2: Simple Synthesis from Relevant Paper

In [5]:
#test_question = df_MedQ['Question'][1]
#final_context, filenames = final_context_builder(graph, test_question, method="hybrid")
filenames = total_tokens = set(spreadsheet.get_worksheet(1).col_values(3)[1:])
print(filenames)

{'HACA', 'AUPTICUPP', 'TRISS', 'PTARDS', 'PROWESS-SHOCK', 'ITMNCA', 'CHEST', 'EDVARCA', 'UHSEPSIS', 'OTCSCA', 'LSPA', 'CORTICUS', 'LOVIT', 'EHPSS', 'DDMLSMVICP', 'PROWESS', 'DLASPEEPvsARDSNET', 'APV', 'ENB', 'SEPSISPAM', 'TTM2', 'ESCNBC', 'EDvsLSMENDS', 'EXACT', 'Rivers-Trial', 'DPSMVAS', 'PDHP', 'SPICE-III', 'SOAP-II', 'NSS', 'MPOHCAUSPITHRCA', 'CEIIUPPSARDS', 'SOPA-III', 'ASD', 'IDEAL-ICU', 'PVOEMVRARDS', 'MHNOCA', 'SAFE', 'ANNANE', 'RDSCI', 'ROSE', 'ARDSNet', 'RARDS', 'FDSD ', 'PPATDSHMA', 'FMWSCPARDS', 'BMIMSARDS', 'EDPMARDSLPMV', 'FEAST', 'DIHRFARDSNET', 'TTM', 'CAC ', 'MIND-USA', 'SADASS', 'SRRCTTHNPCAP', 'THvsTN', 'LCOTARDS', 'ULS', 'PESDCOMV', 'APROCCHSS', 'AMTD', 'DDS', 'ARDSSRDRFMS', 'DPD', 'VASST', 'HARDST', 'HEAT', 'CCCARDS', 'HACA-LA', 'HNPCASR', 'NBSARDS', 'ACURASYS', 'FACTT', 'RUSGATICUD', 'DTNP', 'ALBIOS', 'ETSDMV', 'DIICU', 'ProCESS', 'MCAU ', 'ADRENAL', 'PROSEVA', 'AID-ICU', 'AUDDICU', 'NEvsVP', 'SMDLTSWCI', 'NSEPSIS', 'ARISE', 'SS3vsSS2', 'FRESHS', 'TOF-ARDS', 'TCAIH

In [6]:
# Define model and other parameters at the start
model = "gpt-3.5-turbo"  # or "gpt-4-turbo-preview" if you have access
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Process each file
for file in filenames:
    # Convert file to list if it isn't already
    file_list = [file] if isinstance(file, str) else list(file)
    
    try:
        # Get prompts and context
        system_prompt = load_prompt_template("simple_synthesis")
        firstk_chunks_prompt = get_firstk_chunks(graph, file_list, firstk=100)
        
        prompt = f"""{system_prompt}
        """

        # Make API call
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "system", "content": str(firstk_chunks_prompt)},
            ],
            temperature=0.3,
        )
        
        # Print results
        if response and response.choices:
            print(f"File: {file}")
            print(response.choices[0].message.content)
            print("\n" + "="*50 + "\n")
        else:
            print(f"No response generated for file: {file}\n")
            
    except Exception as e:
        print(f"Error processing file {file}: {str(e)}\n")

File: HACA
{
  "design": "Randomized, controlled trial with blinded assessment",
  "intervention": "Mild therapeutic hypothermia (32-34°C) for 24 hours",
  "population": "275 patients with cardiac arrest due to ventricular fibrillation, aged 18-75 years",
  "outcomes": "55% had a favorable neurologic outcome with hypothermia vs. 39% with normothermia; 14% lower mortality in hypothermia group; no significant difference in complications"
}


File: AUPTICUPP
{
"design": "Literature Review",
"intervention": "Antipsychotics (haloperidol, olanzapine, quetiapine, risperidone)",
"population": "370 children",
"outcomes": "Various antipsychotics used for PICU delirium treatment; no data on prevention; outcomes include relief of agitation and delirium, adverse drug events (e.g., dystonia, hyperpyrexia)"
}


File: TRISS
{
"design": "Multicenter, parallel-group trial",
"intervention": "Transfusion at lower (≤7 g/dL) vs. higher (≤9 g/dL) hemoglobin thresholds",
"population": "998 patients with septi